In [ ]:
!cp /content/drive/MyDrive/School/University-of-Guelph/MDS/Course/26-2S_DATA6700/1_data/data_siamese_gradcam.zip /content/

In [ ]:
!unzip -q /content/data_siamese_gradcam.zip -d /content/data

In [ ]:
import shutil
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rasterio
import torch
from torch.utils.data import DataLoader, Dataset

In [ ]:
from torch.optim.optimizer import T
DIR_DATA_DRIVE = Path("data")
DIR_DATA_LOCAL = Path("/content/data")

# ============================================================
# Directories
# ============================================================
DIR_DATA = DIR_DATA_LOCAL
DIR_METADATA = DIR_DATA_DRIVE / "0_metadata"

DIR_ICEYE = DIR_DATA / "1_ICEYE"
DIR_SENTINEL = DIR_DATA / "2_Sentinel2"
DIR_TERRAIN = DIR_DATA / "3_Terrain" / "depmap"

FILEPATH_PAIR_MANIFEST = DIR_METADATA / "pair-manifest.csv"

# ============================================================
# Dataset
# ============================================================
PATCH_SIZE = 256
NUM_CHANNELS = 3
NUM_CLASSES = 2
METADATA_DIM = 4
CONTROLLED = True

# ============================================================
# DataLoader
# ============================================================
BATCH_SIZE = 32
NUM_WORKERS = 2
PIN_MEMORY = True
SHUFFLE_TRAIN = True
SHUFFLE_VALID = False

# ============================================================
# Training
# ============================================================
EPOCHS = 50
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4
DEVICE = "cuda"

# ============================================================
# Misc
# ============================================================
SEED = 42


In [ ]:
# if not DIR_DATA_LOCAL.exists():
#     print("Copying dataset to local disk...")
#     shutil.copytree(DIR_DATA_DRIVE, DIR_DATA_LOCAL)
#     print("Done!")
# else:
#     print("Dataset already exists.")

In [ ]:
class MetadataNormalizer:
    """
    Normalize continuous metadata and encode categorical metadata.
    """
    def __init__(self):
        self.delta_mean = None
        self.delta_std = None
        self.angle_mean = None
        self.angle_std = None

    def fit(self, pair_manifest):
        """
        Compute normalization statistics from the training manifest.
        """
        train = pair_manifest[pair_manifest["split"] == "train"]

        self.delta_mean = train["delta_days"].mean()
        self.delta_std = train["delta_days"].std()

        self.angle_mean = train["incidence_angle"].mean()
        self.angle_std = train["incidence_angle"].std()

        print("Metadata statistics")
        print(
            f"delta_days : "
            f"{self.delta_mean:.2f} ± {self.delta_std:.2f}"
        )
        print(
            f"incidence_angle : "
            f"{self.angle_mean:.2f} ± {self.angle_std:.2f}"
        )

    def transform(self, sample):
        """
        Convert one sample into a normalized metadata vector.
        """
        delta_days = (
            sample["delta_days"] - self.delta_mean
        ) / self.delta_std

        incidence_angle = (
            sample["incidence_angle"] - self.angle_mean
        ) / self.angle_std

        orbit_direction = (
            1.0
            if sample["orbit_direction"] == "Ascending"
            else 0.0
        )

        look_side = (
            1.0
            if sample["look_side"] == "Right"
            else 0.0
        )

        return np.array(
            [
                delta_days,
                incidence_angle,
                orbit_direction,
                look_side,
            ],
            dtype=np.float32,
        )

class FloodDataset(Dataset):
    def __init__(
        self,
        split: str,
        metadata_normalizer=None,
        api_decay=0.8,
        controlled=False
    ):
        """
        split:
            "train", "valid", or "test"
        """
        self.metadata_normalizer = metadata_normalizer
        self.samples = []
        self.target_suffix = "_controlled" if controlled else ""

        pair_manifest = pd.read_csv(
            DIR_DATA / FILEPATH_PAIR_MANIFEST
        )

        # --------------------------------------------------
        # Filter pairs
        # --------------------------------------------------
        if split not in ("train", "valid", "test"):
            raise ValueError("split must be 'train', 'valid', or 'test'")

        pair_manifest = pair_manifest[pair_manifest["split"] == split]

        # --------------------------------------------------
        # Build sample list
        # --------------------------------------------------
        for _, pair in pair_manifest.iterrows():

            sar_base_dir = DIR_ICEYE / pair["sar_base"]
            sar_target_dir = DIR_ICEYE / pair["sar_target"]

            # Rainfall representation
            rain_history = np.array(
                [
                    pair["precip_-1"],
                    pair["precip_-2"],
                    pair["precip_-3"],
                    pair["precip_-4"],
                    pair["precip_-5"],
                ],
                dtype=np.float32,
            )
            weights = np.array(
                [api_decay ** i for i in range(5)],
                dtype=np.float32,
            )
            api = np.dot(weights, rain_history).astype(np.float32)

            # Every SAR patch becomes one sample
            for sar_path in sorted(sar_base_dir.glob("*.tif")):

                parts = sar_path.stem.split("_", 1)
                if len(parts) != 2:
                    print(f"Unexpected filename: {sar_path.name}")
                    continue

                patch_id = parts[1]

                # Check split
                split_in_patch_id = get_split_from_patch_filename(
                    patch_id
                )
                if split not in split_in_patch_id:
                    continue

                year = get_year_from_patch_id(patch_id)

                base_iceye_id = pair["sar_base"].split("_")[-1]
                target_iceye_id = pair["sar_target"].split("_")[-1]

                sar_target_path = (
                    sar_target_dir
                    / f"{target_iceye_id}_{patch_id}{self.target_suffix}.tif"
                )

                if not sar_target_path.exists():
                    sar_target_path = (
                        sar_target_dir
                        / f"{target_iceye_id}_{patch_id}.tif"
                    )

                ndvi_target_path = (
                    DIR_SENTINEL /
                    f"patches_{year}" /
                    pair["ndvi_target"] /
                    f"{pair['ndvi_target']}_{patch_id}_NDVI{self.target_suffix}.tif"
                )

                if not ndvi_target_path.exists():
                    ndvi_target_path = (
                        DIR_SENTINEL /
                        f"patches_{year}" /
                        pair["ndvi_target"] /
                        f"{pair['ndvi_target']}_{patch_id}_NDVI.tif"
                    )

                sample = {
                    "pair_id": pair["pair_id"],
                    "patch_id": patch_id,

                    "label":
                        1 if pair["pair_id"].startswith("W")
                        else 0,

                    "sar_base":
                        sar_base_dir /
                        f"{base_iceye_id}_{patch_id}.tif",

                    "sar_target":
                        sar_target_path,

                    "ndvi_base":
                        DIR_SENTINEL /
                        f"patches_{year}" /
                        pair["ndvi_base"] /
                        f"{pair['ndvi_base']}_{patch_id}_NDVI.tif",

                    "ndvi_target":
                        ndvi_target_path,

                    "terrain":
                        DIR_TERRAIN /
                        f"patches_{year}" /
                        f"depmap_{patch_id}.tif",

                    # --------------------------
                    # Raw metadata
                    # --------------------------
                    "delta_days":
                        pair["delta_days"],

                    "incidence_angle":
                        pair["incidence_angle"],

                    "orbit_direction":
                        pair["orbit_direction"],

                    "look_side":
                        pair["look_side"],

                    # --------------------------
                    # Rainfall target
                    # --------------------------
                    "rain":
                        api,
                }

                # Skip incomplete samples
                paths = [
                    sample["sar_base"],
                    sample["sar_target"],
                    sample["ndvi_base"],
                    sample["ndvi_target"],
                    sample["terrain"],
                ]

                if all(p.exists() for p in paths):
                    self.samples.append(sample)
                # for p in paths:
                #     if not p.exists():
                #         print(p)
                #         break
                # else:
                #     self.samples.append(sample)

        print(f"{split}: {len(self.samples)} samples")

    def __len__(self):
        return len(self.samples)

    def _read_tiff(self, path):
        with rasterio.open(path) as src:
            img = src.read(1).astype(np.float32)
        return img

    def __getitem__(self, idx):

        sample = self.samples[idx]

        # ----------------------------
        # Load images
        # ----------------------------
        sar_base = self._read_tiff(sample["sar_base"])
        sar_target = self._read_tiff(sample["sar_target"])
        ndvi_base = self._read_tiff(sample["ndvi_base"])
        ndvi_target = self._read_tiff(sample["ndvi_target"])
        terrain = self._read_tiff(sample["terrain"])

        # ----------------------------
        # Build image tensors
        # ----------------------------
        before = np.stack([
            sar_base,
            ndvi_base,
            terrain,
        ])
        after = np.stack([
            sar_target,
            ndvi_target,
            terrain,
        ])
        before = torch.from_numpy(before)
        after = torch.from_numpy(after)

        # ----------------------------
        # Label
        # ----------------------------
        label = torch.tensor(sample["label"], dtype=torch.float32)

        # ----------------------------
        # Metadata
        # ----------------------------
        metadata = self.metadata_normalizer.transform(sample)
        metadata = torch.from_numpy(metadata)

        # ----------------------------
        # Rain target
        # ----------------------------
        rain = torch.tensor(sample["rain"], dtype=torch.float32)

        return {
            "before": before,
            "after": after,
            "label": label,
            "rain": rain,
            "metadata": metadata,
            "pair_id": sample["pair_id"],
            "patch_id": sample["patch_id"],
        }


def get_year_from_patch_id(patch_id: str) -> int:
    """
    Extract the year from a patch ID.

    Examples
    --------
    B24BC0120 -> 2024
    W25CB0120 -> 2025
    """

    return 2000 + int(patch_id[2:4])

from pathlib import Path

def get_split_from_patch_filename(patch_id: str) -> str:
    """
    Extract the dataset split from a SAR patch ID.

    Examples
    --------
    TB24CC0120 -> "train/valid"
    EB24CC0120 -> "test"
    """
    split = patch_id[:1]
    if split == "T":
        return "train/valid"
    elif split == "E":
        return "test"

    raise ValueError(
        f"Unexpected split '{split}' in {patch_id}"
    )


In [ ]:
pair_manifest = pd.read_csv(FILEPATH_PAIR_MANIFEST)

metadata_normalizer = MetadataNormalizer()
metadata_normalizer.fit(pair_manifest)

train_dataset = FloodDataset(
    "train",
    metadata_normalizer=metadata_normalizer,
)

valid_dataset = FloodDataset(
    "valid",
    metadata_normalizer=metadata_normalizer,
)

test_dataset = FloodDataset(
    "test",
    metadata_normalizer=metadata_normalizer,
    controlled=CONTROLLED
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=SHUFFLE_TRAIN,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY
)
valid_loader = DataLoader(
    valid_dataset,
    batch_size=BATCH_SIZE,
    shuffle=SHUFFLE_VALID,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY
)
test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY
)

In [ ]:
sample = test_dataset[13]

print(f"pair_id  : {sample['pair_id']}")
print(f"patch_id : {sample['patch_id']}")
print(f"label    : {sample['label']}")
print(f"metadata : {sample['metadata']}")
print(f"rain     : {sample['rain']}")

print(sample["before"].shape)
print(sample["after"].shape)

CHANNEL_NAMES = [
    "SAR",
    "NDVI",
    "Terrain"
]
fig, axes = plt.subplots(
    2,
    3,
    figsize=(12, 8)
)
for i in range(3):
    axes[0, i].imshow(
        sample["before"][i],
        cmap="gray"
    )
    axes[0, i].set_title(
        f"Before ({CHANNEL_NAMES[i]})"
    )
    axes[0, i].axis("off")
    axes[1, i].imshow(
        sample["after"][i],
        cmap="gray"
    )
    axes[1, i].set_title(
        f"After ({CHANNEL_NAMES[i]})"
    )
    axes[1, i].axis("off")

plt.tight_layout()
plt.show()

In [ ]:
batch = next(iter(train_loader))

print(batch["before"].shape)
print(batch["after"].shape)
print(batch["label"].shape)

In [ ]:
print(torch.isnan(batch["before"]).any())
print(torch.isnan(batch["after"]).any())

In [ ]:
import copy
import time
from datetime import datetime

import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.models as models

In [ ]:
DIR_CHECKPOINTS = Path("/content/drive/MyDrive/School/University-of-Guelph/MDS/Course/26-2S_DATA6700/2_model_checkpoints")
DIR_CHECKPOINTS.mkdir(
    parents=True,
    exist_ok=True
)
CHECKPOINT_NAME_BASE = "siamese_gradcam_cls"

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)
print(f"Device: {DEVICE}")

EPOCHS = 50
BATCH_SIZE = 32
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4

In [ ]:
class MultiScalePooling(nn.Module):
    """
    Multi-scale global pooling using
    Adaptive Average Pooling + Adaptive Max Pooling.
    """

    def __init__(self):
        super().__init__()

        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)

    def forward(self, x):

        avg = self.avg_pool(x)
        mx = self.max_pool(x)

        x = torch.cat([avg, mx], dim=1)

        return x.flatten(1)


class SiameseFloodNetClassification(nn.Module):
    def __init__(
        self,
        metadata_dim=4,
        pretrained=True,
    ):
        super().__init__()

        # --------------------------------------------------
        # Shared ResNet18 encoder
        # --------------------------------------------------
        backbone = models.resnet18(
            weights=(
                models.ResNet18_Weights.DEFAULT
                if pretrained
                else None
            )
        )

        # Replace first convolution
        backbone.conv1 = nn.Conv2d(
            in_channels=3,
            out_channels=64,
            kernel_size=7,
            stride=2,
            padding=3,
            bias=False,
        )

        # Remove average pooling and FC layer
        self.encoder = nn.Sequential(
            backbone.conv1,
            backbone.bn1,
            backbone.relu,
            backbone.maxpool,
            backbone.layer1,
            backbone.layer2,
            backbone.layer3,
            backbone.layer4,
        )

        # --------------------------------------------------
        # Multi-scale pooling
        # --------------------------------------------------
        self.pool = MultiScalePooling()

        # Before (1024) + After (1024)
        image_feature_dim = 512 * 2 * 2

        # --------------------------------------------------
        # Metadata encoder
        # --------------------------------------------------
        self.metadata_encoder = nn.Sequential(
            nn.Linear(metadata_dim, 16),
            nn.ReLU(inplace=True),

            nn.Linear(16, 32),
            nn.ReLU(inplace=True),
        )

        # --------------------------------------------------
        # Classification head
        # --------------------------------------------------
        self.classification_head = nn.Sequential(
            nn.Linear(image_feature_dim + 32, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),

            nn.Linear(512, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(0.2),

            nn.Linear(128, 1),
        )

    def forward(
        self,
        before,
        after,
        metadata,
    ):
        # --------------------------------------------------
        # Shared encoder
        # --------------------------------------------------
        feat_before = self.encoder(before)
        feat_after = self.encoder(after)

        # --------------------------------------------------
        # Multi-scale pooling
        # --------------------------------------------------
        feat_before = self.pool(feat_before)
        feat_after = self.pool(feat_after)

        # --------------------------------------------------
        # Concatenate image features
        # --------------------------------------------------
        image_feature = torch.cat(
            [
                feat_before,
                feat_after,
            ],
            dim=1,
        )

        # --------------------------------------------------
        # Metadata branch
        # --------------------------------------------------
        metadata_feature = self.metadata_encoder(
            metadata
        )

        # --------------------------------------------------
        # Late fusion
        # --------------------------------------------------
        feature = torch.cat(
            [
                image_feature,
                metadata_feature,
            ],
            dim=1,
        )

        # --------------------------------------------------
        # Classification
        # --------------------------------------------------
        logits = self.classification_head(feature)

        return logits.squeeze(1)

In [ ]:
def train_one_epoch(
    model,
    loader,
    criterion,
    optimizer
):
    model.train()

    running_loss = 0.0
    running_correct = 0
    total = 0

    for batch in loader:
        before = batch["before"].to(DEVICE)
        after = batch["after"].to(DEVICE)
        metadata = batch["metadata"].to(DEVICE)

        label = (
            batch["label"]
            .float()
            .to(DEVICE)
        )
        optimizer.zero_grad()

        logits = model(
            before,
            after,
            metadata
        )
        loss = criterion(
            logits,
            label
        )

        loss.backward()
        optimizer.step()

        batch_size = label.size(0)
        running_loss += loss.item() * batch_size

        with torch.no_grad():
            prediction = (torch.sigmoid(logits) >= 0.5)
            running_correct += (prediction == label.bool()).sum().item()
        total += batch_size

    epoch_loss = running_loss / total
    epoch_accuracy = running_correct / total

    return epoch_loss, epoch_accuracy


@torch.no_grad()
def validate(
    model,
    loader,
    criterion
):
    model.eval()

    running_loss = 0.0
    running_correct = 0
    total = 0

    for batch in loader:
        before = batch["before"].to(DEVICE)
        after = batch["after"].to(DEVICE)
        metadata = batch["metadata"].to(DEVICE)

        label = (
            batch["label"]
            .float()
            .to(DEVICE)
        )

        logits = model(
            before,
            after,
            metadata
        )
        loss = criterion(
            logits,
            label
        )

        batch_size = label.size(0)
        running_loss += loss.item() * batch_size

        prediction = logits >= 0
        running_correct += (
            prediction == label.bool()
        ).sum().item()

        total += batch_size

    epoch_loss = running_loss / total
    epoch_accuracy = running_correct / total

    return epoch_loss, epoch_accuracy


In [ ]:
# Initialize model
model = SiameseFloodNetClassification(metadata_dim=METADATA_DIM).to(DEVICE)
print(model)

In [ ]:
# Test model
batch = next(iter(train_loader))

before = batch["before"].to(DEVICE)
after = batch["after"].to(DEVICE)
metadata = batch["metadata"].to(DEVICE)
label = batch["label"].to(DEVICE)

output = model(
    before,
    after,
    metadata
)
print(output.shape)
# torch.Size([32, 3])

In [ ]:
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)
scheduler = optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=EPOCHS
)

In [ ]:
best_valid_accuracy = 0.0
history = []

start = time.time()

for epoch in range(EPOCHS):
    train_loss, train_accuracy = train_one_epoch(
        model,
        train_loader,
        criterion,
        optimizer,
    )
    valid_loss, valid_accuracy = validate(
        model,
        valid_loader,
        criterion,
    )
    scheduler.step()

    # Save epoch results
    history.append({
        "epoch": epoch + 1,
        "train_loss": train_loss,
        "train_accuracy": train_accuracy,
        "valid_loss": valid_loss,
        "valid_accuracy": valid_accuracy,
    })

    print(
        f"Epoch {epoch+1:03d} | "
        f"Train Loss {train_loss:.4f} | "
        f"Train Acc {train_accuracy:.3f} | "
        f"Valid Loss {valid_loss:.4f} | "
        f"Valid Acc {valid_accuracy:.3f}"
    )

    if valid_accuracy > best_valid_accuracy:
        best_valid_accuracy = valid_accuracy
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        model_name = (
            f"{CHECKPOINT_NAME_BASE}_"
            f"{timestamp}_"
            f"epoch{epoch+1:02d}_"
            f"acc{valid_accuracy:.3f}.pth"
        )
        torch.save(
            {
                "epoch": epoch + 1,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "scheduler_state_dict": scheduler.state_dict(),
                "valid_loss": valid_loss,
                "valid_accuracy": valid_accuracy,
            },
            DIR_CHECKPOINTS / model_name,
        )
        print("✓ Best model updated.")

elapsed = time.time() - start

# Convert to DataFrame
log_train_df = pd.DataFrame(history)

print(f"\nTraining finished in {elapsed/60:.1f} minutes.")
print(f"Best validation accuracy: {best_valid_accuracy:.3f}")

In [ ]:
### SiameseFloodNetClassification ###

log = """
Epoch 001 | Train Loss 0.6762 | Train Acc 0.590 | Valid Loss 0.7042 | Valid Acc 0.483
✓ Best model updated.
Epoch 002 | Train Loss 0.5590 | Train Acc 0.721 | Valid Loss 0.7096 | Valid Acc 0.550
✓ Best model updated.
Epoch 003 | Train Loss 0.4221 | Train Acc 0.819 | Valid Loss 0.6324 | Valid Acc 0.617
✓ Best model updated.
Epoch 004 | Train Loss 0.2756 | Train Acc 0.892 | Valid Loss 0.5222 | Valid Acc 0.728
✓ Best model updated.
Epoch 005 | Train Loss 0.1521 | Train Acc 0.942 | Valid Loss 0.8002 | Valid Acc 0.661
Epoch 006 | Train Loss 0.1094 | Train Acc 0.964 | Valid Loss 0.9522 | Valid Acc 0.628
Epoch 007 | Train Loss 0.0782 | Train Acc 0.974 | Valid Loss 0.9193 | Valid Acc 0.706
Epoch 008 | Train Loss 0.0582 | Train Acc 0.979 | Valid Loss 1.1954 | Valid Acc 0.628
Epoch 009 | Train Loss 0.0423 | Train Acc 0.986 | Valid Loss 0.5483 | Valid Acc 0.794
✓ Best model updated.
Epoch 010 | Train Loss 0.0294 | Train Acc 0.992 | Valid Loss 0.6532 | Valid Acc 0.794
Epoch 011 | Train Loss 0.0219 | Train Acc 0.993 | Valid Loss 0.7969 | Valid Acc 0.789
Epoch 012 | Train Loss 0.0211 | Train Acc 0.994 | Valid Loss 0.7155 | Valid Acc 0.778
Epoch 013 | Train Loss 0.0431 | Train Acc 0.985 | Valid Loss 0.7270 | Valid Acc 0.711
Epoch 014 | Train Loss 0.0454 | Train Acc 0.986 | Valid Loss 1.4326 | Valid Acc 0.628
Epoch 015 | Train Loss 0.0685 | Train Acc 0.976 | Valid Loss 0.5653 | Valid Acc 0.772
Epoch 016 | Train Loss 0.0277 | Train Acc 0.992 | Valid Loss 0.9033 | Valid Acc 0.739
Epoch 017 | Train Loss 0.0193 | Train Acc 0.994 | Valid Loss 1.2991 | Valid Acc 0.667
Epoch 018 | Train Loss 0.0087 | Train Acc 0.997 | Valid Loss 0.8585 | Valid Acc 0.728
Epoch 019 | Train Loss 0.0099 | Train Acc 0.997 | Valid Loss 0.6727 | Valid Acc 0.789
Epoch 020 | Train Loss 0.0084 | Train Acc 0.996 | Valid Loss 0.8799 | Valid Acc 0.744
Epoch 021 | Train Loss 0.0105 | Train Acc 0.997 | Valid Loss 0.8188 | Valid Acc 0.789
Epoch 022 | Train Loss 0.0154 | Train Acc 0.993 | Valid Loss 0.9614 | Valid Acc 0.750
Epoch 023 | Train Loss 0.0119 | Train Acc 0.993 | Valid Loss 0.8362 | Valid Acc 0.756
Epoch 024 | Train Loss 0.0029 | Train Acc 1.000 | Valid Loss 0.6712 | Valid Acc 0.806
✓ Best model updated.
Epoch 025 | Train Loss 0.0037 | Train Acc 1.000 | Valid Loss 0.7534 | Valid Acc 0.772
Epoch 026 | Train Loss 0.0019 | Train Acc 1.000 | Valid Loss 0.8162 | Valid Acc 0.783
Epoch 027 | Train Loss 0.0102 | Train Acc 0.996 | Valid Loss 0.6054 | Valid Acc 0.800
Epoch 028 | Train Loss 0.0066 | Train Acc 0.997 | Valid Loss 0.8514 | Valid Acc 0.800
Epoch 029 | Train Loss 0.0089 | Train Acc 0.996 | Valid Loss 0.5745 | Valid Acc 0.828
✓ Best model updated.
Epoch 030 | Train Loss 0.0015 | Train Acc 1.000 | Valid Loss 0.6073 | Valid Acc 0.817
Epoch 031 | Train Loss 0.0015 | Train Acc 1.000 | Valid Loss 0.6431 | Valid Acc 0.800
Epoch 032 | Train Loss 0.0021 | Train Acc 0.999 | Valid Loss 0.6885 | Valid Acc 0.800
Epoch 033 | Train Loss 0.0005 | Train Acc 1.000 | Valid Loss 0.7058 | Valid Acc 0.817
Epoch 034 | Train Loss 0.0060 | Train Acc 0.999 | Valid Loss 0.6037 | Valid Acc 0.822
Epoch 035 | Train Loss 0.0019 | Train Acc 1.000 | Valid Loss 0.7413 | Valid Acc 0.811
Epoch 036 | Train Loss 0.0017 | Train Acc 1.000 | Valid Loss 0.7188 | Valid Acc 0.811
Epoch 037 | Train Loss 0.0012 | Train Acc 1.000 | Valid Loss 0.6647 | Valid Acc 0.822
Epoch 038 | Train Loss 0.0011 | Train Acc 1.000 | Valid Loss 0.7192 | Valid Acc 0.806
Epoch 039 | Train Loss 0.0029 | Train Acc 0.999 | Valid Loss 0.6938 | Valid Acc 0.811
Epoch 040 | Train Loss 0.0015 | Train Acc 1.000 | Valid Loss 0.7026 | Valid Acc 0.794
Epoch 041 | Train Loss 0.0007 | Train Acc 1.000 | Valid Loss 0.6885 | Valid Acc 0.794
Epoch 042 | Train Loss 0.0006 | Train Acc 1.000 | Valid Loss 0.7269 | Valid Acc 0.789
Epoch 043 | Train Loss 0.0009 | Train Acc 1.000 | Valid Loss 0.6860 | Valid Acc 0.806
Epoch 044 | Train Loss 0.0007 | Train Acc 1.000 | Valid Loss 0.6938 | Valid Acc 0.789
Epoch 045 | Train Loss 0.0003 | Train Acc 1.000 | Valid Loss 0.7180 | Valid Acc 0.794
Epoch 046 | Train Loss 0.0005 | Train Acc 1.000 | Valid Loss 0.7115 | Valid Acc 0.806
Epoch 047 | Train Loss 0.0005 | Train Acc 1.000 | Valid Loss 0.6907 | Valid Acc 0.789
Epoch 048 | Train Loss 0.0011 | Train Acc 1.000 | Valid Loss 0.7099 | Valid Acc 0.800
Epoch 049 | Train Loss 0.0010 | Train Acc 1.000 | Valid Loss 0.7458 | Valid Acc 0.806
Epoch 050 | Train Loss 0.0006 | Train Acc 1.000 | Valid Loss 0.7047 | Valid Acc 0.800
"""

# Training finished in 12.7 minutes.
# Best validation accuracy: 0.828

In [ ]:
import re

import pandas as pd
import matplotlib.pyplot as plt
from pydantic import BaseModel

class EpochMetrics(BaseModel):
    epoch: int
    train_loss: float
    train_acc: float
    valid_loss: float
    valid_acc: float

pattern = re.compile(
    r"Epoch\s+(\d+)"
    r"\s+\|\s+Train Loss\s+([\d.]+)"
    r"\s+\|\s+Train Acc\s+([\d.]+)"
    r"\s+\|\s+Valid Loss\s+([\d.]+)"
    r"\s+\|\s+Valid Acc\s+([\d.]+)"
)

history: list[EpochMetrics] = []

for match in pattern.finditer(log):
    epoch, train_loss, train_acc, valid_loss, valid_acc = match.groups()

    history.append(
        EpochMetrics(
            epoch=epoch,
            train_loss=train_loss,
            train_acc=train_acc,
            valid_loss=valid_loss,
            valid_acc=valid_acc,
        )
    )

log_train_df = pd.DataFrame(
    x.model_dump()
    for x in history
)

In [ ]:
best_idx_loss = log_train_df["valid_loss"].idxmin()
best_epoch_loss = log_train_df.loc[best_idx_loss, "epoch"]
best_loss = log_train_df.loc[best_idx_loss, "valid_loss"]

plt.figure(figsize=(8, 5))

plt.plot(
    log_train_df["epoch"],
    log_train_df["train_loss"],
    label="Training"
)

plt.plot(
    log_train_df["epoch"],
    log_train_df["valid_loss"],
    label="Validation"
)

plt.scatter(
    best_epoch_loss,
    best_loss,
    color="red",
    zorder=3,
    label=f"Best validation ({best_epoch_loss})"
)

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training and Validation Loss (Classification)")

plt.legend()
plt.grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
best_idx_acc = log_train_df["valid_acc"].idxmax()
best_epoch_acc = log_train_df.loc[best_idx_acc, "epoch"]
best_acc = log_train_df.loc[best_idx_acc, "valid_acc"]

plt.figure(figsize=(8, 5))

plt.plot(
    log_train_df["epoch"],
    log_train_df["train_acc"],
    label="Training"
)

plt.plot(
    log_train_df["epoch"],
    log_train_df["valid_acc"],
    label="Validation"
)

plt.scatter(
    best_epoch_acc,
    best_acc,
    color="red",
    zorder=3,
    label=f"Best validation ({best_epoch_acc})"
)

plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Training and Validation Accuracy (Classification)")

plt.legend()
plt.grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
model = SiameseFloodNetClassification(metadata_dim=METADATA_DIM).to(DEVICE)
checkpoint = torch.load(
    DIR_CHECKPOINTS / "siamese_gradcam_cls_20260722_204620_epoch29_acc0.828.pth",
    map_location=DEVICE
)
model.load_state_dict(
    checkpoint["model_state_dict"]
)
model.eval()

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
)

In [ ]:
@torch.no_grad()
def evaluate_test(model, loader):
    model.eval()

    y_true = []
    y_pred = []
    y_prob = []

    for batch in loader:

        before = batch["before"].to(DEVICE)
        after = batch["after"].to(DEVICE)
        metadata = batch["metadata"].to(DEVICE)

        label = batch["label"].cpu().numpy()

        logits = model(
            before,
            after,
            metadata
        )

        probability = torch.sigmoid(logits)
        prediction = (probability >= 0.5)

        y_true.append(label)
        y_pred.append(prediction.cpu().numpy())
        y_prob.append(probability.cpu().numpy())

    y_true = np.concatenate(y_true).astype(int)
    y_pred = np.concatenate(y_pred).astype(int)
    y_prob = np.concatenate(y_prob)

    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(
        y_true,
        y_pred,
        zero_division=0
    )
    recall = recall_score(
        y_true,
        y_pred,
        zero_division=0
    )
    f1 = f1_score(
        y_true,
        y_pred,
        zero_division=0
    )

    cm = confusion_matrix(
        y_true,
        y_pred
    )

    print(f"Accuracy : {accuracy:.3f}")
    print(f"Precision: {precision:.3f}")
    print(f"Recall   : {recall:.3f}")
    print(f"F1-score : {f1:.3f}")

    print("\nConfusion Matrix")
    print(cm)

    return y_true, y_pred, y_prob

In [ ]:
y_true, y_pred, y_prob = evaluate_test(
    model,
    test_loader
)

In [ ]:
### Normal ###
# Accuracy : 0.867
# Precision: 0.867
# Recall   : 0.867
# F1-score : 0.867

# Confusion Matrix
# [[26  4]
#  [ 4 26]]

### Controlled ###
# Accuracy : 0.783
# Precision: 0.840
# Recall   : 0.700
# F1-score : 0.764

# Confusion Matrix
# [[26  4]
#  [ 9 21]]

In [ ]:
!pip install grad-cam

In [ ]:
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import numpy as np

In [ ]:
class AfterBranchWrapper(nn.Module):
    def __init__(self, model, before, metadata):
        super().__init__()
        self.model = model
        self.before = before
        self.metadata = metadata

    def forward(self, after):
        output = self.model(
            self.before,
            after,
            self.metadata
        )

        # Grad-CAM expects shape (batch_size, num_outputs)
        return output.unsqueeze(1)


def show_gradcam(
    model,
    dataset,
    index,
    device=DEVICE
):
    """
    Display Grad-CAM for one sample.

    Parameters
    ----------
    model : nn.Module
        Trained Siamese classification model.
    dataset : FloodDataset
    index : int
        Sample index.
    """
    sample = dataset[index]

    before = sample["before"].unsqueeze(0).to(device)
    after = sample["after"].unsqueeze(0).to(device)
    metadata = sample["metadata"].unsqueeze(0).to(device)

    # --------------------------------------------------
    # Prediction
    # --------------------------------------------------
    model.eval()

    with torch.no_grad():
        logit = model(
            before,
            after,
            metadata
        )

        probability = torch.sigmoid(logit).item()
        prediction = int(probability >= 0.5)

    ground_truth = int(sample["label"].item())

    # --------------------------------------------------
    # Images
    # --------------------------------------------------
    before_img = before.squeeze().cpu().numpy()[0]
    after_img = after.squeeze().cpu().numpy()[0]

    before_img = (
        before_img - before_img.min()
    ) / (
        before_img.max() - before_img.min() + 1e-6
    )

    after_img = (
        after_img - after_img.min()
    ) / (
        after_img.max() - after_img.min() + 1e-6
    )

    # --------------------------------------------------
    # Grad-CAM
    # --------------------------------------------------
    wrapped_model = AfterBranchWrapper(
        model,
        before,
        metadata
    )

    cam = GradCAM(
        model=wrapped_model,
        target_layers=[model.encoder[-1]]
    )

    grayscale_cam = cam(
        input_tensor=after,
        targets=[ClassifierOutputTarget(0)]
    )[0]

    # --------------------------------------------------
    # Plot
    # --------------------------------------------------
    fig, ax = plt.subplots(
        2,
        2,
        figsize=(10, 10),
        constrained_layout=True
    )

    ax[0, 0].imshow(before_img, cmap="gray")
    ax[0, 0].set_title("SAR Before")
    ax[0, 0].axis("off")

    ax[0, 1].imshow(after_img, cmap="gray")
    ax[0, 1].set_title("SAR After")
    ax[0, 1].axis("off")

    ax[1, 0].imshow(before_img, cmap="gray")

    im = ax[1, 0].imshow(
        grayscale_cam,
        cmap="jet",
        alpha=0.4,
        vmin=0,
        vmax=1
    )

    ax[1, 0].set_title("Grad-CAM")
    ax[1, 0].axis("off")

    ax[1, 1].imshow(after_img, cmap="gray")
    ax[1, 1].imshow(
        grayscale_cam,
        cmap="jet",
        alpha=0.4,
        vmin=0,
        vmax=1
    )

    ax[1, 1].set_title("Grad-CAM")
    ax[1, 1].axis("off")

    fig.colorbar(
        im,
        ax=ax,
        shrink=0.7,
        label="Importance"
    )

    correct = prediction == ground_truth

    fig.suptitle(
        f"Pair : {sample['pair_id']}\n"
        f"Patch: {sample['patch_id']}\n"
        f"Predicted probability : {probability:.3f}\n"
        f"Prediction : {prediction}\n"
        f"Ground Truth : {ground_truth}\n"
        f"{'Correct' if correct else 'Incorrect'}",
        fontsize=13,
    )

    plt.show()

In [ ]:
show_gradcam(
    model,
    test_dataset,
    24,
)

In [ ]:
from pytorch_grad_cam import LayerCAM
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget

class AfterBranchWrapper(nn.Module):
    def __init__(self, model, before, metadata):
        super().__init__()
        self.model = model
        self.before = before
        self.metadata = metadata

    def forward(self, after):
        output = self.model(
            self.before,
            after,
            self.metadata
        )
        return output.unsqueeze(1)


def show_layercam(
    model,
    dataset,
    index,
    target_layers=None,
    device=DEVICE,
):
    """
    Display Layer-CAM for one sample.

    Parameters
    ----------
    model : nn.Module
        Trained Siamese classification model.

    dataset : FloodDataset

    index : int
        Sample index.

    target_layers : list[nn.Module] or None
        Encoder layer(s) to visualize.
        If None, uses the last encoder layer.
    """

    sample = dataset[index]

    before = sample["before"].unsqueeze(0).to(device)
    after = sample["after"].unsqueeze(0).to(device)
    metadata = sample["metadata"].unsqueeze(0).to(device)

    model.eval()

    with torch.no_grad():
        logit = model(
            before,
            after,
            metadata
        )

        probability = torch.sigmoid(logit).item()
        prediction = int(probability >= 0.5)

    ground_truth = int(sample["label"].item())

    # --------------------------------------------------
    # Images
    # --------------------------------------------------

    before_img = before.squeeze().cpu().numpy()[0]
    after_img = after.squeeze().cpu().numpy()[0]

    before_img = (
        before_img - before_img.min()
    ) / (
        before_img.max() - before_img.min() + 1e-6
    )

    after_img = (
        after_img - after_img.min()
    ) / (
        after_img.max() - after_img.min() + 1e-6
    )

    # --------------------------------------------------
    # Layer selection
    # --------------------------------------------------

    if target_layers is None:
        target_layers = [model.encoder[-1]]

    elif not isinstance(target_layers, list):
        target_layers = [target_layers]

    # --------------------------------------------------
    # Layer-CAM
    # --------------------------------------------------

    wrapped_model = AfterBranchWrapper(
        model,
        before,
        metadata
    )

    cam = LayerCAM(
        model=wrapped_model,
        target_layers=target_layers
    )

    grayscale_cam = cam(
        input_tensor=after,
        targets=[ClassifierOutputTarget(0)]
    )[0]

    # --------------------------------------------------
    # Plot
    # --------------------------------------------------

    fig, ax = plt.subplots(
        2,
        2,
        figsize=(10, 10),
        constrained_layout=True
    )

    ax[0, 0].imshow(before_img, cmap="gray")
    ax[0, 0].set_title("SAR Before")
    ax[0, 0].axis("off")

    ax[0, 1].imshow(after_img, cmap="gray")
    ax[0, 1].set_title("SAR After")
    ax[0, 1].axis("off")

    ax[1, 0].imshow(before_img, cmap="gray")

    im = ax[1, 0].imshow(
        grayscale_cam,
        cmap="jet",
        alpha=0.4,
        vmin=0,
        vmax=1,
    )

    ax[1, 0].set_title("Layer-CAM")
    ax[1, 0].axis("off")

    ax[1, 1].imshow(after_img, cmap="gray")
    ax[1, 1].imshow(
        grayscale_cam,
        cmap="jet",
        alpha=0.4,
        vmin=0,
        vmax=1,
    )

    ax[1, 1].set_title("Layer-CAM")
    ax[1, 1].axis("off")

    fig.colorbar(
        im,
        ax=ax,
        shrink=0.7,
        label="Importance",
    )

    correct = prediction == ground_truth

    layer_names = ", ".join(
        [type(layer).__name__ for layer in target_layers]
    )

    fig.suptitle(
        f"Pair : {sample['pair_id']}\n"
        f"Patch: {sample['patch_id']}\n"
        f"Predicted probability : {probability:.3f}\n"
        f"Prediction : {prediction}\n"
        f"Ground Truth : {ground_truth}\n"
        f"Layers : {layer_names}\n"
        f"{'Correct' if correct else 'Incorrect'}",
        fontsize=13,
    )

    plt.show()

In [ ]:
show_layercam(
    model,
    test_dataset,
    21,
    target_layers=[
        model.encoder[-3],
        model.encoder[-2],
        model.encoder[-1],
    ],
)

In [ ]:
from pathlib import Path
import rasterio
import numpy as np
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.model_targets import BinaryClassifierOutputTarget

In [ ]:
DIR_GRADCAM = DIR_DATA_DRIVE / "3_results" / "siamese-gradcam-classification" / "patches-gradcam"
DIR_GRADCAM.mkdir(parents=True, exist_ok=True)

In [ ]:
def export_gradcam_geotiffs(
    model,
    dataset,
    output_dir,
    device=DEVICE,
    controlled=False,
):
    """
    Export Grad-CAM GeoTIFFs for every sample.

    Output filename:
        gradcam_{pair_id}_{patch_id}_api{prediction}.tif
    """
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    suffix = "_controlled" if controlled else ""

    model.eval()

    target_layers = [model.encoder[-1]]

    for idx in range(len(dataset)):

        tensor_sample = dataset[idx]
        file_sample = dataset.samples[idx]

        before = tensor_sample["before"].unsqueeze(0).to(device)
        after = tensor_sample["after"].unsqueeze(0).to(device)
        metadata = tensor_sample["metadata"].unsqueeze(0).to(device)

        # --------------------------------------------------
        # Prediction
        # --------------------------------------------------
        with torch.no_grad():
            logit = model(
                before,
                after,
                metadata
            )
            probability = torch.sigmoid(logit).item()

        # --------------------------------------------------
        # Grad-CAM
        # --------------------------------------------------
        wrapped_model = AfterBranchWrapper(
            model,
            before,
            metadata
        )

        cam = GradCAM(
            model=wrapped_model,
            target_layers=target_layers
        )

        grayscale_cam = cam(
            input_tensor=after,
            targets=[ClassifierOutputTarget(0)]
        )[0]

        # --------------------------------------------------
        # Save GeoTIFF
        # --------------------------------------------------
        with rasterio.open(file_sample["sar_target"]) as src:

            profile = src.profile.copy()
            profile.update(
                driver="GTiff",
                dtype="float32",
                count=1,
                compress="LZW"
            )

            output_path = (
                output_dir /
                f"gradcam_"
                f"{tensor_sample['pair_id']}_"
                f"{tensor_sample['patch_id']}_"
                f"prob{probability:.3f}"
                f"{suffix}.tif"
            )

            with rasterio.open(
                output_path,
                "w",
                **profile
            ) as dst:
                dst.write(
                    grayscale_cam.astype(np.float32),
                    1
                )

        print(
            f"[{idx+1:3d}/{len(dataset)}] "
            f"{output_path.name}"
        )

    print("Done!")

In [ ]:
export_gradcam_geotiffs(
    model,
    test_dataset,
    DIR_GRADCAM,
    controlled=CONTROLLED
)